In [426]:
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer



In [427]:
df = pd.read_csv('/home/alexandre/Projects/ML/KS/Titanic/data/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [428]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [429]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [430]:
X = X.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

In [431]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


In [432]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

In [433]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 643 entries, 868 to 829
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    643 non-null    int64  
 1   Sex       643 non-null    object 
 2   Age       512 non-null    float64
 3   SibSp     643 non-null    int64  
 4   Parch     643 non-null    int64  
 5   Fare      643 non-null    float64
 6   Embarked  641 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 40.2+ KB


In [434]:
OHE = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OHE.fit(X_train[['Sex', 'Embarked']])

X_train_ohe = OHE.transform(X_train[['Sex', 'Embarked']])
X_val_ohe = OHE.transform(X_val[['Sex', 'Embarked']])
X_test_ohe = OHE.transform(X_test[['Sex', 'Embarked']])   


X_train_ohe_df = pd.DataFrame(X_train_ohe, columns = OHE.get_feature_names_out(['Sex', 'Embarked']), index=X_train.index)
X_val_ohe_df = pd.DataFrame(X_val_ohe, columns = OHE.get_feature_names_out(['Sex', 'Embarked']), index=X_val.index)
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns = OHE.get_feature_names_out(['Sex', 'Embarked']), index=X_test.index)


X_train_final = pd.concat([X_train.drop(columns=['Sex', 'Embarked']), X_train_ohe_df], axis=1)
X_val_final   = pd.concat([X_val.drop(columns=['Sex', 'Embarked']), X_val_ohe_df], axis=1)
X_test_final  = pd.concat([X_test.drop(columns=['Sex', 'Embarked']), X_test_ohe_df], axis=1)

print(X_train_final.shape, X_val_final.shape, X_test_final.shape)


(643, 11) (114, 11) (134, 11)


In [435]:
X_train_final.info()


<class 'pandas.core.frame.DataFrame'>
Index: 643 entries, 868 to 829
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        643 non-null    int64  
 1   Age           512 non-null    float64
 2   SibSp         643 non-null    int64  
 3   Parch         643 non-null    int64  
 4   Fare          643 non-null    float64
 5   Sex_female    643 non-null    float64
 6   Sex_male      643 non-null    float64
 7   Embarked_C    643 non-null    float64
 8   Embarked_Q    643 non-null    float64
 9   Embarked_S    643 non-null    float64
 10  Embarked_nan  643 non-null    float64
dtypes: float64(8), int64(3)
memory usage: 60.3 KB


In [436]:
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train_final[['Age']])
X_train_final[['Age']] = imputer.transform(X_train_final[['Age']])
X_val_final[['Age']] = imputer.transform(X_val_final[['Age']])
X_test_final[['Age']] = imputer.transform(X_test_final[['Age']])

In [437]:
X_train_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 643 entries, 868 to 829
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        643 non-null    int64  
 1   Age           643 non-null    float64
 2   SibSp         643 non-null    int64  
 3   Parch         643 non-null    int64  
 4   Fare          643 non-null    float64
 5   Sex_female    643 non-null    float64
 6   Sex_male      643 non-null    float64
 7   Embarked_C    643 non-null    float64
 8   Embarked_Q    643 non-null    float64
 9   Embarked_S    643 non-null    float64
 10  Embarked_nan  643 non-null    float64
dtypes: float64(8), int64(3)
memory usage: 60.3 KB


In [438]:
my_model = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss', early_stopping_rounds=5, random_state=42)

my_model.fit(X_train_final, y_train, 
             eval_set=[(X_val_final, y_val)],
             verbose=True)
preds = my_model.predict(X_test_final)
print("Accuracy:", accuracy_score(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
print("Classification Report:\n", classification_report(y_test, preds))

[0]	validation_0-logloss:0.64709
[1]	validation_0-logloss:0.62525
[2]	validation_0-logloss:0.60559
[3]	validation_0-logloss:0.58800
[4]	validation_0-logloss:0.57237
[5]	validation_0-logloss:0.55742


/home/alexandre/Projects/ML/MLvenv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [16:09:41] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[6]	validation_0-logloss:0.54478
[7]	validation_0-logloss:0.53235
[8]	validation_0-logloss:0.52101
[9]	validation_0-logloss:0.51119
[10]	validation_0-logloss:0.50203
[11]	validation_0-logloss:0.49385
[12]	validation_0-logloss:0.48624
[13]	validation_0-logloss:0.47904
[14]	validation_0-logloss:0.47229
[15]	validation_0-logloss:0.46587
[16]	validation_0-logloss:0.45998
[17]	validation_0-logloss:0.45457
[18]	validation_0-logloss:0.44894
[19]	validation_0-logloss:0.44459
[20]	validation_0-logloss:0.43994
[21]	validation_0-logloss:0.43580
[22]	validation_0-logloss:0.43235
[23]	validation_0-logloss:0.42880
[24]	validation_0-logloss:0.42542
[25]	validation_0-logloss:0.42298
[26]	validation_0-logloss:0.42102
[27]	validation_0-logloss:0.41942
[28]	validation_0-logloss:0.41838
[29]	validation_0-logloss:0.41573
[30]	validation_0-logloss:0.41467
[31]	validation_0-logloss:0.41416
[32]	validation_0-logloss:0.41315
[33]	validation_0-logloss:0.41226
[34]	validation_0-logloss:0.41175
[35]	validation_0-

In [439]:
test = pd.read_csv("/home/alexandre/Projects/ML/KS/Titanic/data/test.csv")


In [440]:
PId = test['PassengerId']

test = test.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

In [441]:
test_ohe = OHE.transform(test[['Sex', 'Embarked']])
test_ohe = pd.DataFrame(test_ohe, columns = OHE.get_feature_names_out(['Sex', 'Embarked']), index=test.index)
test_final  = pd.concat([test.drop(columns=['Sex', 'Embarked']), test_ohe], axis=1)

In [442]:
test_final[['Age']] = imputer.transform(test_final[['Age']])

In [443]:
y_pred = my_model.predict(test_final)
submission = pd.DataFrame({
    "PassengerId": PId,
    "Survived": y_pred.astype(int)  # s'assurer d'avoir 0/1
})

submission.to_csv("/home/alexandre/Projects/ML/KS/Titanic/results /submission_XGBoost.csv", index=False)
print("Fichier écrit → submission.csv")

Fichier écrit → submission.csv
